# EDGAR Report Fetcher
Fetches 10-K annual reports from SEC EDGAR.

In [13]:
# Install dependencies 
# !pip install requests beautifulsoup4

!pip install -r ../requirements.txt

In [14]:
import requests
import time
import re
from bs4 import BeautifulSoup

# Need an email for EDGAR to work 
HEADERS = {"User-Agent": "hauglv@usi.ch"}

COMPANY_CIKS = {
    "JPMorgan": "0000019617",
    "GoldmanSachs": "0000886982",
    "BankofAmerica": "0000070858",
    "WellsFargo": "0000072971",
    "Citigroup": "0000831001",
    "MorganStanley": "0000895421",
    "USBancorp": "0000036104",
    "PNCFinancial": "0000713676",
    "BlackRock": "0001364742",
    "StateStreet": "0000093751",
    "CharlesSchwab": "0000316709",
    "BerkshireHathaway": "0001067983",
    "TRowePrice": "0001113169",
    "MetLife": "0001099219",
    "PrudentialFinancial": "0001137774",
    "Aflac": "0000004977",
    "Allstate": "0000040729",
    "Travelers": "0000086312",
    "Visa": "0001403161",
    "Mastercard": "0001141788",
    "PayPal": "0001633917",
    "Fiserv": "0000798354",
    "AmericanExpress": "0000004962",
    "CapitalOne": "0000927628",
    "DiscoverFinancial": "0001393612",
    "NYSEGroupICE": "0001571949",
    "NasdaqInc": "0001120193",
    "CMEGroup": "0001156375"
}

print("Libraries loaded.")


Libraries loaded.


## Step 1 — Getting the list of 10-K filings

In [15]:
def get_10k_filings(cik: str) -> list:
    """Returns a list of recent 10-K filings for a company."""
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    filings    = data.get("filings", {}).get("recent", {})
    forms      = filings.get("form", [])
    dates      = filings.get("filingDate", [])
    accessions = filings.get("accessionNumber", [])

    results = []
    for form, date, accession in zip(forms, dates, accessions):
        if form == "10-K":
            results.append({"filingDate": date, "accessionNumber": accession})

    return results

print("Function defined.")


Function defined.


## Step 2 — Finding the correct document URL

EDGAR wraps HTML files in an Inline XBRL viewer (`/ix?doc=/Archives/...`).
We strip that prefix so we download the raw HTML directly.

In [16]:
def clean_url(url: str) -> str:
    """
    EDGAR sometimes wraps URLs like:
      https://www.sec.gov/ix?doc=/Archives/edgar/data/.../file.htm
    We strip the /ix?doc= part to get the direct file URL.
    """
    if "/ix?doc=" in url:
        url = "https://www.sec.gov" + url.split("/ix?doc=")[-1]
    return url


def get_readable_doc_url(cik: str, accession: str) -> str:
    """
    Fetches the filing index and finds the best readable HTML document.
    Returns the direct URL (no JavaScript wrapper).
    """
    cik_int   = int(cik)
    acc_clean = accession.replace("-", "")

    index_url = (
        f"https://www.sec.gov/Archives/edgar/data/{cik_int}/"
        f"{acc_clean}/{accession}-index.htm"
    )
    print(f"  Checking index: {index_url}")

    r = requests.get(index_url, headers=HEADERS)
    if r.status_code != 200:
        print(f"  Warning: index returned status {r.status_code}")
        return None

    soup = BeautifulSoup(r.content, "html.parser")
    candidates = []

    for row in soup.find_all("tr"):
        cells = row.find_all("td")
        if len(cells) < 3:
            continue

        link_tag = cells[2].find("a") if len(cells) > 2 else None
        doc_type = cells[3].get_text(strip=True) if len(cells) > 3 else ""
        desc     = cells[1].get_text(strip=True).lower() if len(cells) > 1 else ""

        if not link_tag:
            continue

        href     = link_tag.get("href", "")
        filename = href.split("/")[-1].lower().split("?")[0]  # strip ?doc= params

        # Skip machine-readable files
        skip = [".xml", ".xsd", ".json", "r2.htm", "r3.htm",
                "r4.htm", "r5.htm", "_cal", "_def", "_lab", "_pre"]
        if any(p in filename for p in skip):
            continue

        if not (filename.endswith(".htm") or filename.endswith(".html")):
            continue

        score = 0
        if doc_type.upper() in ("10-K", "10-K/A"): score += 20
        if "annual report" in desc:                 score += 10
        if "10-k" in desc:                          score += 5
        score -= len(filename)  # shorter names preferred

        # Build full URL and strip any XBRL viewer wrapper
        if href.startswith("/"):
            full_url = "https://www.sec.gov" + href
        else:
            full_url = href
        full_url = clean_url(full_url)

        candidates.append((score, full_url, filename))

    if not candidates:
        print("  No readable HTML files found.")
        return None

    candidates.sort(reverse=True)
    print(f"  Selected: {candidates[0][2]}")
    print(f"  Direct URL: {candidates[0][1]}")
    return candidates[0][1]

print("Functions defined.")


Functions defined.


## Step 3 — Downloading and extracting plain text

In [17]:
def fetch_10k_text(cik: str, max_filings: int = 1) -> list:
    """
    Downloads readable text from the most recent 10-K filings.
    Returns a list of dicts: {filingDate, text}
    """
    filings = get_10k_filings(cik)
    results = []

    for filing in filings[:max_filings]:
        print(f"\n  Filing date: {filing['filingDate']}")

        doc_url = get_readable_doc_url(cik, filing["accessionNumber"])
        if not doc_url:
            print("  Skipping — no readable document found.")
            continue

        time.sleep(0.6)

        print(f"  Downloading: {doc_url}")
        r = requests.get(doc_url, headers=HEADERS)

        if r.status_code != 200:
            print(f"  Warning: download failed (status {r.status_code})")
            continue

        # Check if we didn't get the JS viewer page
        if "enable javascript" in r.text.lower()[:300]:
            print("  Warning: still got JavaScript page — check the URL above.")
            continue

        soup = BeautifulSoup(r.content, "html.parser")
        for tag in soup(["script", "style", "head", "nav"]):
            tag.decompose()

        text = soup.get_text(separator=" ", strip=True)
        text = re.sub(r"\s{3,}", "  ", text)

        print(f"  Extracted {len(text):,} characters.")

        if len(text) < 10_000:
            print("  Text seems short, may not be the full report.")
        else:
            print("  Text looks good.")

        results.append({"filingDate": filing["filingDate"], "text": text})
        time.sleep(0.6)

    return results

print("Function defined.")


Function defined.


## Run — Fetching a report and preview

In [18]:
company = "JPMorgan"
cik     = COMPANY_CIKS[company]

print(f"Fetching 10-K for {company}...\n")
filings = fetch_10k_text(cik, max_filings=1)

for f in filings:
    print(f"\nFiling date : {f['filingDate']}")
    print(f"Text length : {len(f['text']):,} characters")
    print(f"\nPreview:")
    print(f['text'][:800])


Fetching 10-K for JPMorgan...


  Filing date: 2026-02-13
  Checking index: https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/0001628280-26-008131-index.htm
  Selected: jpm-20251231.htm
  Direct URL: https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm
  Downloading: https://www.sec.gov/Archives/edgar/data/19617/000162828026008131/jpm-20251231.htm
  Extracted 1,410,471 characters.
  Text looks good.

Filing date : 2026-02-13
Text length : 1,410,471 characters

Preview:
0000019617 FALSE 2025 FY true false June 30, 2026 230 June 30, 2026 228 June 30, 2026 229 August 7, 2026 270 June 30, 2026 230 June 30, 2026 230 June 30, 2026 228 June 30, 2026 228 June 30, 2026 229 June 30, 2026 229 http://fasb.org/us-gaap/2025#PrincipalTransactionsRevenue http://fasb.org/us-gaap/2025#PrincipalTransactionsRevenue http://fasb.org/us-gaap/2025#PrincipalTransactionsRevenue http://fasb.org/us-gaap/2025#TradingLiabilities http://fasb.org/us-gaap/2025#TradingLiabili